# Линейные модели и бустинг для регрессии

Сначала короткий разбор на игрушечных данных (линейная регрессия → деревья и бустинг для регрессии), затем **самостоятельная работа** с реальным биологическим датасетом.

| Блок | Содержание | Время |
|---|---|---|
| 0 | Подготовка | 2 мин |
| 1 | Линейная регрессия: что внутри, нелинейность, регуляризация, мультиколлинеарность | ~15 мин |
| 2 | Деревья и бустинг для регрессии, отличие от линейных моделей | ~15 мин |
| 3 | **Самостоятельная работа:** Abalone — возраст моллюска по замерам | ~55 мин |

Ноутбук рассчитан на Google Colab (Runtime → Run all, GPU не нужен).

## 0. Подготовка

In [ ]:
import importlib.util, subprocess, sys

for pkg in ["xgboost", "lightgbm"]:
    if importlib.util.find_spec(pkg) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (train_test_split, KFold, cross_val_score, validation_curve,
                                     GridSearchCV, RandomizedSearchCV)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.compose import TransformedTargetRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso, RidgeCV, LassoCV
from sklearn.tree import DecisionTreeRegressor, export_text
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.inspection import permutation_importance
import lightgbm as lgb
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

SEED = 42
np.random.seed(SEED)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
GRAY, BLUE, ORANGE, GREEN, VIOLET, RED = "#8a8984", "#2a78d6", "#eb6834", "#1baf7a", "#6250d6", "#e34948"


def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def metrics(y_true, y_pred):
    return {"RMSE": rmse(y_true, y_pred), "MAE": float(mean_absolute_error(y_true, y_pred)),
            "R2": float(r2_score(y_true, y_pred))}


CV = KFold(n_splits=5, shuffle=True, random_state=SEED)
CV_TABLE = {}          # сюда складываем результаты кросс-валидации: имя -> (RMSE среднее, std)


def cv_rmse(model, X, y, name=None):
    """5-fold CV: RMSE (среднее, std по фолдам). Если задано name — запоминает в CV_TABLE."""
    s = -cross_val_score(model, X, y, cv=CV, scoring="neg_root_mean_squared_error", n_jobs=-1)
    if name:
        CV_TABLE[name] = (s.mean(), s.std())
    return s.mean(), s.std()

## 1. Линейная регрессия

$$\hat y = w_0 + w_1 x_1 + \dots + w_p x_p,\qquad \min_w \frac1n\sum_i (y_i-\hat y_i)^2 .$$

Из лекции: минимизация MSE — это метод максимального правдоподобия при гауссовом шуме. Начнём с игрушечных данных: **скорость роста проростков в зависимости от температуры** (синтетика: оптимум около 27 °C).

In [ ]:
rng = np.random.default_rng(SEED)
n = 150
temp = rng.uniform(5, 40, n)
growth = 10 * np.exp(-((temp - 27) / 8) ** 2) + rng.normal(0, 0.8, n)      # колокол + шум

X_all, y_all = temp.reshape(-1, 1), growth
X_tr, X_te, y_tr, y_te = train_test_split(X_all, y_all, test_size=0.3, random_state=SEED)
grid = np.linspace(5, 40, 400).reshape(-1, 1)

fig, ax = plt.subplots(figsize=(7, 3.8))
ax.scatter(X_tr, y_tr, c=BLUE, s=22, label="train")
ax.scatter(X_te, y_te, c=ORANGE, s=22, label="test")
ax.set_xlabel("температура, °C"); ax.set_ylabel("скорость роста, мм/сут"); ax.legend(frameon=False)
plt.show()

In [ ]:
lin = LinearRegression().fit(X_tr, y_tr)
print(f"w0 = {lin.intercept_:.3f}, w1 = {lin.coef_[0]:.3f}")
print("train:", {k: round(v, 3) for k, v in metrics(y_tr, lin.predict(X_tr)).items()})
print("test: ", {k: round(v, 3) for k, v in metrics(y_te, lin.predict(X_te)).items()})

fig, ax = plt.subplots(figsize=(7, 3.8))
ax.scatter(X_tr, y_tr, c=GRAY, s=18)
ax.plot(grid, lin.predict(grid), c=BLUE, lw=2.5)
ax.set_title("Линейная модель не видит оптимума: недообучение", loc="left")
plt.show()

### 1.1 Что внутри `LinearRegression`

Для MSE есть решение в явном виде (нормальные уравнения) $w=(X^\top X)^{-1}X^\top y$. Но можно и градиентным спуском — как в лекции. Проверим, что всё три способа дают одно и то же.

In [ ]:
# 1) нормальные уравнения
Xd = np.c_[np.ones(len(X_tr)), X_tr]
w_closed = np.linalg.solve(Xd.T @ Xd, Xd.T @ y_tr)

# 2) градиентный спуск по стандартизованному признаку
mu, sd = X_tr.mean(), X_tr.std()
z = (X_tr[:, 0] - mu) / sd
w0, w1, lr, hist = 0.0, 0.0, 0.1, []
for step in range(200):
    err = (w0 + w1 * z) - y_tr
    hist.append(np.mean(err ** 2))
    w0 -= lr * 2 * err.mean()
    w1 -= lr * 2 * (err * z).mean()
w_gd = np.array([w0 - w1 * mu / sd, w1 / sd])         # обратно в исходный масштаб

print("нормальные уравнения:", w_closed.round(4))
print("градиентный спуск:   ", w_gd.round(4))
print("sklearn:             ", np.array([lin.intercept_, lin.coef_[0]]).round(4))

fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(hist, c=BLUE); ax.set_yscale("log"); ax.set_xlabel("шаг"); ax.set_ylabel("MSE на train")
plt.show()

### 1.2 Линейная модель ≠ прямая линия: полиномиальные признаки

Модель линейна **по весам**, а признаки можно преобразовывать: $x \to (x, x^2, \dots, x^d)$. Растим степень и смотрим на train/test.

In [ ]:
degrees = [1, 2, 3, 4, 6, 8, 12, 15]
rows = []
for d in degrees:
    m = make_pipeline(StandardScaler(), PolynomialFeatures(d, include_bias=False), LinearRegression()).fit(X_tr, y_tr)
    rows.append({"степень": d, "train RMSE": rmse(y_tr, m.predict(X_tr)), "test RMSE": rmse(y_te, m.predict(X_te))})
res = pd.DataFrame(rows).set_index("степень")
display(res.round(3))

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
axes[0].plot(res.index, res["train RMSE"], "o-", c=BLUE, label="train")
axes[0].plot(res.index, res["test RMSE"], "o-", c=ORANGE, label="test")
axes[0].set_yscale("log"); axes[0].set_xlabel("степень полинома"); axes[0].set_ylabel("RMSE"); axes[0].legend(frameon=False)
axes[1].scatter(X_tr, y_tr, c=GRAY, s=16)
for d, c in [(1, BLUE), (4, GREEN), (15, RED)]:
    m = make_pipeline(StandardScaler(), PolynomialFeatures(d, include_bias=False), LinearRegression()).fit(X_tr, y_tr)
    axes[1].plot(grid, m.predict(grid), c=c, lw=2, label=f"степень {d}")
axes[1].set_ylim(-4, 14); axes[1].legend(frameon=False)
plt.tight_layout(); plt.show()

### 1.3 Регуляризация лечит переобучение

Оставим степень 15, но добавим L2-штраф (Ridge). Силу штрафа $\alpha$ выбираем кросс-валидацией — как в лекции.

In [ ]:
pipe = make_pipeline(StandardScaler(), PolynomialFeatures(15, include_bias=False), StandardScaler(), Ridge())
gs = GridSearchCV(pipe, {"ridge__alpha": np.logspace(-4, 3, 15)}, cv=5, scoring="neg_root_mean_squared_error").fit(X_tr, y_tr)
ridge_poly = gs.best_estimator_
print("лучший alpha:", gs.best_params_["ridge__alpha"])
print("test RMSE, степень 15 + Ridge:", round(rmse(y_te, ridge_poly.predict(X_te)), 3))

fig, ax = plt.subplots(figsize=(7, 3.8))
ax.scatter(X_tr, y_tr, c=GRAY, s=16)
ax.plot(grid, ridge_poly.predict(grid), c=GREEN, lw=2.5)
ax.set_ylim(-4, 14); ax.set_title("Полином степени 15 + Ridge", loc="left")
plt.show()

### 1.4 Мультиколлинеарность: коэффициенты «прыгают»

Два почти одинаковых признака (например, длина и ширина зерна). Истина: $y = x_1 + x_2 + \varepsilon$. Обучим модель на 200 бутстрэп-выборках и посмотрим на веса $(w_1, w_2)$.

In [ ]:
rng = np.random.default_rng(SEED)
m_ = 100
x1 = rng.normal(size=m_)
x2 = x1 + rng.normal(0, 0.1, m_)                     # почти копия x1
y_c = x1 + x2 + rng.normal(0, 1, m_)
Xc = np.c_[x1, x2]
print("corr(x1, x2) =", round(np.corrcoef(x1, x2)[0, 1], 3))

models = {"OLS": LinearRegression(), "Ridge": Ridge(alpha=5.0), "Lasso": Lasso(alpha=0.1)}
W = {k: [] for k in models}
for _ in range(200):
    idx = rng.integers(0, m_, m_)
    for k, mdl in models.items():
        W[k].append(mdl.fit(Xc[idx], y_c[idx]).coef_.copy())

fig, axes = plt.subplots(1, 3, figsize=(13, 3.8), sharex=True, sharey=True)
for ax, (k, col) in zip(axes, [("OLS", BLUE), ("Ridge", GREEN), ("Lasso", ORANGE)]):
    w = np.array(W[k]); ax.scatter(w[:, 0], w[:, 1], s=10, c=col, alpha=.6)
    ax.axhline(1, c=GRAY, lw=.8); ax.axvline(1, c=GRAY, lw=.8)
    ax.set_title(f"{k}: std(w1) = {w[:, 0].std():.2f}", loc="left"); ax.set_xlabel("w1")
axes[0].set_ylabel("w2")
plt.tight_layout(); plt.show()

**Вывод:** OLS находит сумму $w_1+w_2\approx 2$, но делит её между признаками случайно — интерпретировать отдельные веса нельзя. Ridge делит поровну и стабилизирует, Lasso выбирает один признак и обнуляет другой.

## 2. Деревья и бустинг для регрессии

В регрессии лист дерева предсказывает **среднее** $y$ объектов в нём, а разбиение выбирается по максимальному снижению MSE. Посмотрим на те же данные.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.6), sharey=True)
for ax, d in zip(axes, [1, 3, None]):
    t = DecisionTreeRegressor(max_depth=d, random_state=SEED).fit(X_tr, y_tr)
    ax.scatter(X_tr, y_tr, c=GRAY, s=14)
    ax.plot(grid, t.predict(grid), c=ORANGE, lw=2.2)
    ax.set_title(f"max_depth = {d}: test RMSE {rmse(y_te, t.predict(X_te)):.2f}", loc="left")
plt.tight_layout(); plt.show()

t2 = DecisionTreeRegressor(max_depth=2, random_state=SEED).fit(X_tr, y_tr)
print(export_text(t2, feature_names=["temp"]))

Предсказание дерева — «лесенка». Глубина 1 — недообучение, без ограничения глубины — запоминание (по листу на объект). Лечат это **ансамблем**: лес усредняет независимые глубокие деревья (бэггинг), бустинг последовательно строит мелкие деревья.

In [ ]:
rf = RandomForestRegressor(n_estimators=300, min_samples_leaf=3, random_state=SEED, n_jobs=-1).fit(X_tr, y_tr)
gb = GradientBoostingRegressor(n_estimators=200, learning_rate=0.05, max_depth=2, random_state=SEED).fit(X_tr, y_tr)
tree = DecisionTreeRegressor(max_depth=4, random_state=SEED).fit(X_tr, y_tr)

fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(X_tr, y_tr, c=GRAY, s=14)
for mdl, name, c in [(ridge_poly, "полином 15 + Ridge", BLUE), (tree, "дерево (глубина 4)", ORANGE),
                     (rf, "случайный лес", GREEN), (gb, "градиентный бустинг", VIOLET)]:
    ax.plot(grid, mdl.predict(grid), c=c, lw=2, label=name)
ax.set_ylim(-3, 13); ax.legend(frameon=False); plt.show()

tab = pd.DataFrame({name: metrics(y_te, mdl.predict(X_te)) for name, mdl in
                    [("линейная", lin), ("полином+Ridge", ridge_poly), ("дерево", tree), ("лес", rf), ("бустинг", gb)]}).T
display(tab.round(3))

### 2.1 Бустинг изнутри: деревья учатся на остатках

Для MSE антиградиент — это обычные остатки. Повторим три итерации вручную и сравним с `GradientBoostingRegressor`.

In [ ]:
lr_, M = 0.3, 3
gb3 = GradientBoostingRegressor(n_estimators=M, learning_rate=lr_, max_depth=2, random_state=SEED).fit(X_tr, y_tr)

pred = np.full(len(y_tr), y_tr.mean())               # F0 = константа
for m in range(M):
    resid = y_tr - pred                               # остатки = антиградиент MSE
    h = DecisionTreeRegressor(max_depth=2, criterion="friedman_mse", random_state=SEED).fit(X_tr, resid)
    pred = pred + lr_ * h.predict(X_tr)
    print(f"итерация {m+1}: train RMSE = {rmse(y_tr, pred):.4f}")
print("sklearn, 3 итерации:    ", round(rmse(y_tr, gb3.predict(X_tr)), 4))
print("макс. расхождение предсказаний:", np.abs(pred - gb3.predict(X_tr)).max())

In [ ]:
gb_demo = GradientBoostingRegressor(n_estimators=100, learning_rate=0.3, max_depth=2, random_state=SEED).fit(X_tr, y_tr)
staged = list(gb_demo.staged_predict(grid))
fig, axes = plt.subplots(1, 5, figsize=(17, 3), sharey=True)
for ax, m in zip(axes, [1, 2, 5, 20, 100]):
    ax.scatter(X_tr, y_tr, c=GRAY, s=8)
    ax.plot(grid, staged[m - 1], c=VIOLET, lw=2); ax.set_title(f"{m} деревьев", loc="left")
plt.tight_layout(); plt.show()

### 2.2 Главное отличие от линейных моделей: деревья не экстраполируют

Обучим всё на $x\in[0,7]$ и посмотрим на $x\in[7,14]$ (данные идут по линейному тренду).

In [ ]:
rng = np.random.default_rng(SEED)
xe = rng.uniform(0, 14, 200)
ye = 2 + 0.8 * xe + rng.normal(0, 0.5, 200)
msk = xe < 7
Xe_tr, ye_tr = xe[msk].reshape(-1, 1), ye[msk]
gx = np.linspace(0, 14, 300).reshape(-1, 1)

fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(xe[msk], ye[msk], c=GRAY, s=14, label="train")
ax.scatter(xe[~msk], ye[~msk], c="#c9c8c4", s=14, label="новая область")
for mdl, name, c in [(LinearRegression(), "линейная", BLUE),
                     (RandomForestRegressor(200, random_state=SEED), "лес", GREEN),
                     (GradientBoostingRegressor(random_state=SEED), "бустинг", VIOLET)]:
    ax.plot(gx, mdl.fit(Xe_tr, ye_tr).predict(gx), c=c, lw=2, label=name)
ax.axvline(7, c=RED, ls="--", lw=1); ax.legend(frameon=False); plt.show()

### 2.3 Итог и функции потерь в библиотеках

| | Линейные | Деревья / бустинг |
|---|---|---|
| Нелинейность и взаимодействия | только если добавить вручную | из коробки |
| Масштабирование признаков | нужно (особенно с регуляризацией) | не нужно |
| Мультиколлинеарность | веса нестабильны → Ridge/Lasso | почти не мешает |
| Экстраполяция | да | нет (плато) |
| Интерпретация | веса | importance, PDP |
| Гиперпараметры | $\alpha$, степень | глубина, `learning_rate`, число деревьев… |

Регрессионные функции потерь: sklearn `loss="squared_error" / "absolute_error" / "huber"`; XGBoost `objective="reg:squarederror" / "reg:absoluteerror" / "reg:pseudohubererror"`; LightGBM `objective="regression" / "regression_l1" / "huber"`.

## 3. Самостоятельная работа: возраст моллюска (Abalone)

**Контекст.** Возраст морского ушка (*abalone*) определяют по кольцам на раковине: раковину распиливают, красят и считают кольца под микроскопом — долго и трудоёмко. Зато размеры и массы измерить легко. **Задача: предсказать число колец (`rings`; возраст ≈ rings + 1.5 года) по замерам.**

Датасет: UCI Abalone, 4177 моллюсков.

| Признак | Описание |
|---|---|
| `sex` | M / F / I (infant, молодь) |
| `length`, `diameter`, `height` | размеры раковины, мм (масштабированы) |
| `whole_weight` | масса целого моллюска, г |
| `shucked_weight` | масса мяса, г |
| `viscera_weight` | масса внутренностей, г |
| `shell_weight` | масса высушенной раковины, г |
| `rings` | **целевая переменная** — число колец |

In [ ]:
UCI_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/abalone/abalone.data"
LOCAL_ABALONE_PATH = None      # если нет интернета: загрузите abalone.data в Colab и укажите путь, напр. "/content/abalone.data"
COLS = ["sex", "length", "diameter", "height", "whole_weight", "shucked_weight",
        "viscera_weight", "shell_weight", "rings"]


def load_abalone():
    if LOCAL_ABALONE_PATH is None:
        try:                                                  # вариант 1: OpenML
            from sklearn.datasets import fetch_openml
            d = fetch_openml(data_id=183, as_frame=True).frame
            if d.shape != (4177, 9):
                raise ValueError(f"неожиданный размер {d.shape}")
            d.columns = COLS
        except Exception as e:
            print("OpenML недоступен, пробуем UCI:", repr(e)[:150])
            d = pd.read_csv(UCI_URL, header=None, names=COLS)
    else:
        d = pd.read_csv(LOCAL_ABALONE_PATH, header=None, names=COLS)
    d["sex"] = d["sex"].astype(str)
    d["rings"] = d["rings"].astype(float)
    return d


df = load_abalone()
print("Размер:", df.shape)
df.head()

**Правила работы**

- Целевая переменная — `rings`. Метрики: **RMSE** (основная), MAE, R².
- Модели выбираем только **кросс-валидацией на train**. Тест трогаем один раз — в задании 8.
- Готовые помощники: `rmse`, `metrics`, `cv_rmse(model, X, y, name)` (результат сохраняется в `CV_TABLE`).
- Пишите короткий вывод (1–2 предложения) после каждого задания.